# Query Snowflake-Managed Iceberg Tables from Apache Spark

This notebook connects Spark 4.0 to Snowflake-managed Iceberg tables through Horizon Catalog.
No data is copied — Spark reads the open Iceberg files directly.

**Prerequisites:**
- Conda environment: `conda activate fleet-spark`
- Packages: `pip install pyspark==4.0.0 jupyter python-dotenv`

In [1]:
import os
from dotenv import load_dotenv
load_dotenv('config.env')

# Configuration
SNOWFLAKE_ACCOUNT_URL = os.getenv('SNOWFLAKE_ACCOUNT', '').replace('.', '-')
SNOWFLAKE_PAT = os.getenv('SNOWFLAKE_PAT', '')
SNOWFLAKE_DATABASE = os.getenv('SNOWFLAKE_DATABASE', 'FLEET_DB')
SNOWFLAKE_ROLE = os.getenv('SNOWFLAKE_ROLE', 'ACCOUNTADMIN')

ICEBERG_VERSION = '1.10.1'
SCALA_VERSION = '2.13'

print(f'Account: {SNOWFLAKE_ACCOUNT_URL}')
print(f'Database: {SNOWFLAKE_DATABASE}')

Account: TNEULKY-JT38090
Database: FLEET_DB


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('Fleet Analytics - Iceberg Interop') \
    .master('local[*]') \
    .config('spark.driver.host', '127.0.0.1') \
    .config('spark.driver.bindAddress', '127.0.0.1') \
    .config('spark.jars.packages',
            f'org.apache.iceberg:iceberg-spark-runtime-4.0_{SCALA_VERSION}:{ICEBERG_VERSION},'
            f'org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION}') \
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions') \
    .config('spark.sql.catalog.horizon', 'org.apache.iceberg.spark.SparkCatalog') \
    .config('spark.sql.catalog.horizon.type', 'rest') \
    .config('spark.sql.catalog.horizon.uri',
            f'https://{SNOWFLAKE_ACCOUNT_URL}.snowflakecomputing.com/polaris/api/catalog') \
    .config('spark.sql.catalog.horizon.credential', SNOWFLAKE_PAT) \
    .config('spark.sql.catalog.horizon.warehouse', SNOWFLAKE_DATABASE) \
    .config('spark.sql.catalog.horizon.scope', f'session:role:{SNOWFLAKE_ROLE}') \
    .config('spark.sql.catalog.horizon.header.X-Iceberg-Access-Delegation',
            'vended-credentials') \
    .config('spark.sql.iceberg.vectorization.enabled', 'false') \
    .getOrCreate()

print(f'Spark {spark.version} connected to Horizon Catalog!')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/02 08:46:24 WARN Utils: Your hostname, codespaces-17ba55, resolves to a loopback address: 127.0.0.1; using 10.0.11.62 instead (on interface eth0)
26/11/02 08:46:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/conda/envs/fleet-spark/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/codespace/.ivy2.5.2/cache
The jars for the packages stored in: /home/codespace/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-59042a1e-dfd5-4b44-83a6-3d3f64a96ec5;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.10.1 i

Spark 4.0.0 connected to Horizon Catalog!


## List Tables

All Iceberg tables (including Dynamic Iceberg Tables) are visible through Horizon Catalog.

In [3]:
spark.sql('SHOW TABLES IN horizon.RAW').show(truncate=False)
spark.sql('SHOW TABLES IN horizon.ANALYTICS').show(truncate=False)

26/11/02 08:49:12 WARN AuthManagers: Inferring rest.auth.type=oauth2 since property credential was provided. Please explicitly set rest.auth.type to avoid this warning.
26/11/02 08:49:14 WARN OAuth2Manager: Iceberg REST client is missing the OAuth2 server URI configuration and defaults to https://TNEULKY-JT38090.snowflakecomputing.com/polaris/api/catalog/v1/oauth/tokens. This automatic fallback will be removed in a future Iceberg release. It is recommended to configure the OAuth2 endpoint using the 'oauth2-server-uri' property to be prepared. This warning will disappear if the OAuth2 endpoint is explicitly configured. See https://github.com/apache/iceberg/issues/10537


+---------+------------------------+-----------+
|namespace|tableName               |isTemporary|
+---------+------------------------+-----------+
|RAW      |MAINTENANCE_LOGS        |false      |
|RAW      |SENSOR_READINGS         |false      |
|RAW      |VEHICLE_REGISTRY        |false      |
|RAW      |VEHICLE_TELEMETRY_STREAM|false      |
+---------+------------------------+-----------+

+---------+-------------------+-----------+
|namespace|tableName          |isTemporary|
+---------+-------------------+-----------+
|ANALYTICS|DAILY_FLEET_SUMMARY|false      |
|ANALYTICS|TELEMETRY_ENRICHED |false      |
+---------+-------------------+-----------+



## Query Structured Data

Read the vehicle registry — standard columnar data.

In [4]:
spark.sql("""
    SELECT VEHICLE_ID, MAKE, MODEL, YEAR, FLEET_REGION
    FROM horizon.RAW.VEHICLE_REGISTRY
    LIMIT 10
""").show()

[Stage 0:>                                                          (0 + 1) / 1]

+----------+------------+---------+----+-----------------+
|VEHICLE_ID|        MAKE|    MODEL|YEAR|     FLEET_REGION|
+----------+------------+---------+----+-----------------+
|   VH-0000|        Ford|  Transit|2020|Pacific Northwest|
|   VH-0001|   Chevrolet|  Express|2021|       California|
|   VH-0002|      Toyota|   Tacoma|2022|    Mountain West|
|   VH-0003|         Ram|ProMaster|2023|          Midwest|
|   VH-0004|Freightliner| Cascadia|2024|        Northeast|
|   VH-0005|        Ford|  Transit|2025|Pacific Northwest|
|   VH-0006|   Chevrolet|  Express|2020|       California|
|   VH-0007|      Toyota|   Tacoma|2021|    Mountain West|
|   VH-0008|         Ram|ProMaster|2022|          Midwest|
|   VH-0009|Freightliner| Cascadia|2023|        Northeast|
+----------+------------+---------+----+-----------------+



## Query VARIANT Data with `variant_get()`

Spark 4.0 supports the Iceberg V3 VARIANT type natively. Use `variant_get()` to extract nested fields from JSON telemetry.

In [5]:
df = spark.sql("""
    SELECT
        VEHICLE_ID,
        EVENT_TIMESTAMP,
        variant_get(TELEMETRY_DATA, '$.speed_mph', 'float') AS speed_mph,
        variant_get(TELEMETRY_DATA, '$.engine.temperature_f', 'int') AS engine_temp,
        variant_get(TELEMETRY_DATA, '$.location.lat', 'float') AS latitude,
        variant_get(TELEMETRY_DATA, '$.location.lon', 'float') AS longitude
    FROM horizon.RAW.VEHICLE_TELEMETRY_STREAM
    WHERE variant_get(TELEMETRY_DATA, '$.speed_mph', 'float') > 60
    LIMIT 10
""")
df.show()

26/11/02 08:51:16 WARN CorruptStatistics: Ignoring statistics because created_by is null or empty! See PARQUET-251 and PARQUET-297
                                                                                

+----------+--------------------+---------+-----------+---------+-----------+
|VEHICLE_ID|     EVENT_TIMESTAMP|speed_mph|engine_temp| latitude|  longitude|
+----------+--------------------+---------+-----------+---------+-----------+
|   VH-0010|2026-11-02 07:40:...|     73.7|        208|47.409428| -121.90367|
|   VH-0015|2026-11-02 07:35:...|     68.4|        200|47.609314|-122.123215|
|   VH-0000|2026-11-02 07:35:...|     63.6|        199|47.800423| -122.55929|
|   VH-0038|2026-11-02 07:36:...|     72.6|        203|41.943012|  -87.61646|
|   VH-0018|2026-11-02 07:36:...|     63.6|        185|41.981403|  -88.03916|
|   VH-0014|2026-11-02 07:36:...|     61.5|        197|41.119965| -73.673744|
|   VH-0019|2026-11-02 07:36:...|     62.5|        206|  40.7715|  -74.14206|
|   VH-0015|2026-11-02 07:35:...|     60.3|        195| 47.60821| -122.12385|
|   VH-0000|2026-11-02 07:35:...|     65.0|        199|47.800247| -122.55906|
|   VH-0025|2026-11-02 07:35:...|     62.8|        213|47.600098

## Query Dynamic Iceberg Tables

Dynamic Iceberg Tables are just regular Iceberg tables to external engines.

In [6]:
spark.sql("""
    SELECT *
    FROM horizon.ANALYTICS.DAILY_FLEET_SUMMARY
    ORDER BY summary_date DESC
    LIMIT 10
""").show()

[Stage 4:>                                                          (0 + 1) / 1]

+------------+-----------------+------------+---------------+-------------+-----------------+------------------+---------------+-------------------+
|SUMMARY_DATE|     FLEET_REGION|TOTAL_EVENTS|ACTIVE_VEHICLES|AVG_SPEED_MPH|AVG_ENGINE_TEMP_F|AVG_FUEL_LEVEL_PCT|CRITICAL_EVENTS|CHECK_ENGINE_EVENTS|
+------------+-----------------+------------+---------------+-------------+-----------------+------------------+---------------+-------------------+
|  2026-11-02|    Mountain West|         118|             10|         35.4|            204.5|              70.5|              4|                 15|
|  2026-11-02|        Northeast|         117|             10|         33.7|            204.6|              47.0|              1|                  0|
|  2026-11-02|       California|         116|             10|         24.4|            198.8|              49.2|              0|                 29|
|  2026-11-02|          Midwest|         118|             10|         35.1|            196.7|             

## Aggregate Sensor Readings

Run analytics on the sensor readings table.

In [7]:
spark.sql("""
    SELECT
        VEHICLE_ID,
        COUNT(*) AS reading_count,
        ROUND(AVG(ENGINE_TEMP_F), 1) AS avg_engine_temp,
        ROUND(AVG(FUEL_CONSUMPTION_GPH), 2) AS avg_fuel_gph
    FROM horizon.RAW.SENSOR_READINGS
    GROUP BY VEHICLE_ID
    ORDER BY avg_fuel_gph DESC
    LIMIT 10
""").show()

+----------+-------------+---------------+------------+
|VEHICLE_ID|reading_count|avg_engine_temp|avg_fuel_gph|
+----------+-------------+---------------+------------+
|   VH-0092|          100|          204.7|       11.86|
|   VH-0013|          100|          189.2|       11.45|
|   VH-0091|          100|          172.5|       10.16|
|   VH-0095|          100|          155.0|        9.34|
|   VH-0081|          100|          205.5|        8.24|
|   VH-0052|          100|          175.9|        8.17|
|   VH-0048|          100|          174.0|        8.01|
|   VH-0003|          100|          223.3|        7.94|
|   VH-0077|          100|          209.8|         7.8|
|   VH-0014|          100|          151.7|        7.58|
+----------+-------------+---------------+------------+



In [8]:
spark.stop()
print('Done! Spark read Snowflake-managed Iceberg tables with zero data movement.')

Done! Spark read Snowflake-managed Iceberg tables with zero data movement.
